# CodeGuard AI · Experiment 01: Which datasets, and why?

**Goal:** explain every dataset the project uses, what one row means, where the labels come from, and why the scanner's ML model is trained on **Python** data.

**Steps**
1. What a vulnerability dataset looks like
2. Devign (C) — what is in it
3. CVEfixes (Python) — what is in it
4. Other datasets we considered
5. Our own test fixtures
6. Side-by-side comparison and the language decision

## Step 1: What a vulnerability dataset looks like

To teach a model to spot vulnerable code, we need many code samples, each with a **label**:

| Code sample | Label |
|---|---|
| a function that builds SQL by gluing strings | 1 = vulnerable |
| the same function using parameters | 0 = safe |

Researchers usually get labels from **security-fix commits**: when a developer fixes a CVE, the code *before* the fix is labelled vulnerable and the code *after* the fix is labelled fixed. This is cheap and realistic, but **noisy**: a fix commit may also touch unrelated functions, which then get the wrong label.

Four questions we ask about every dataset:
1. **Language** — does it match what our scanner reads?
2. **Unit** — is one row a function, a file, or a line?
3. **Label source** — how was "vulnerable" decided?
4. **Groups** — which rows are related (same commit, same project)? Related rows must stay on the same side of a train/test split.

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path("../datasets/raw")
PROCESSED = Path("../datasets/processed")

## Step 2: Devign (C)

- **Paper:** Zhou et al., *Devign*, NeurIPS 2019. Distributed as the CodeXGLUE "defect detection" task.
- **Code:** C functions from two projects, **QEMU** (an emulator) and **FFmpeg** (audio/video).
- **Labels:** security-fix commits were checked by hand; functions changed by a fix are labelled vulnerable.
- **Download:** `python scripts/download_devign.py`.

In [2]:
devign = pd.read_parquet(RAW / "devign_raw.parquet")
print("Rows:", len(devign))
print("Columns:", list(devign.columns))
print()
print("Share vulnerable by project:")
print(devign.groupby("project")["target"].agg(["size", "mean"]).round(3))
print()
print("Distinct commits:", devign["commit_id"].nunique())

Rows: 27318
Columns: ['id', 'func', 'target', 'project', 'commit_id', 'split']

Share vulnerable by project:
          size   mean
project              
FFmpeg    9769  0.510
qemu     17549  0.426

Distinct commits: 12316


In [3]:
example = devign.iloc[0]
print("project:", example["project"], "| vulnerable:", example["target"])
print(example["func"][:400])

project: FFmpeg | vulnerable: False
static av_cold int vdadec_init(AVCodecContext *avctx)

{

    VDADecoderContext *ctx = avctx->priv_data;

    struct vda_context *vda_ctx = &ctx->vda_ctx;

    OSStatus status;

    int ret;



    ctx->h264_initialized = 0;



    /* init pix_fmts of codec */

    if (!ff_h264_vda_decoder.pix_fmts) {

        if (kCFCoreFoundationVersionNumber < kCFCoreFoundationVersionNumber10_7)

            ff


**What we learn:** Devign is large and well known, but it is **C**. Our scanner and Bandit read **Python**. A model trained on C cannot be presented as a Python vulnerability detector, so Devign is used only to practise and document the method (Experiments 02–03).

## Step 3: CVEfixes (Python subset)

- **Paper:** Bhandari, Naseer & Moonen, *CVEfixes: Automated Collection of Vulnerabilities and Their Fixes from Open-Source Software*, PROMISE 2021.
- **Download:** Zenodo, DOI 10.5281/zenodo.4476563 (v1.0.8, July 2024; a 12.7 GB zip). Licence: CC BY 4.0.
- **Size (whole dataset, all languages):** 11,873 CVEs, 12,107 fix commits, 4,249 projects, 272 CWE types, 138,974 changed functions.
- **Structure:** a relational database linking CVE → fix commit → changed file → changed method.
  - `method_change`: one row per method version, with its `code` and `before_change` (True = before the fix).
  - `file_change`: the changed file, including its `programming_language`.
  - `fixes`: which commit fixes which CVE, and the repository URL.
  - `cwe_classification`: the weakness type (CWE) of each CVE.
- **Our labels:** method before the fix = **1 (vulnerable)**, method after the fix = **0 (fixed)**.
- **Groups for splitting:** the repository, so a project never appears in both train and test.
- **Build:** `python scripts/build_python_dataset.py --db datasets/raw/CVEfixes.db` → `datasets/processed/python_functions.csv`.

In [4]:
python_csv = PROCESSED / "python_functions.csv"
if python_csv.exists():
    py = pd.read_csv(python_csv)
    print("Python methods:", len(py))
    print(py["target"].value_counts().rename({1: "vulnerable (before fix)", 0: "fixed (after fix)"}))
    print("Repositories:", py["repo_url"].nunique(), "| CVEs:", py["cve_id"].nunique())
else:
    print("Not built yet: download CVEfixes and run scripts/build_python_dataset.py (see Documents/Datasets_Guide.docx).")

Not built yet: download CVEfixes and run scripts/build_python_dataset.py (see Documents/Datasets_Guide.docx).


**Why CVEfixes for the scanner:** it is real-world Python, the labels come from real CVE fixes, every row has a CWE so we can measure recall per weakness type, and the repository field lets us split without leakage.

**What makes it hard:** the vulnerable and fixed versions of a method are often almost identical (the fix may be one line). A word-counting model sees nearly the same words in both. Expect modest scores and report them honestly.

## Step 4: Other datasets we considered

| Dataset | Language | Why not the main choice |
|---|---|---|
| **Big-Vul** (Fan et al., MSR 2020) | C/C++ | Same language problem as Devign. |
| **VUDENC** (Wartschinski et al., 2022) | Python | Good fallback: Python commits for 7 vulnerability types (SQL injection, XSS, command injection, …). Smaller and labelled at token level, so it needs more conversion work. |
| **DetectVul/CVEFixes** (Hugging Face) | Python | Small (5.7k rows) and quick to download, but rows are statement lists with undocumented label codes, so we do not use it for training. |
| **Juliet / NIST SARD** | C, C++, Java, C# | Synthetic test cases per CWE. Useful for controlled tests, but no Python. |

## Step 5: Our own test fixtures

Hand-written Python files in `tests/fixtures/` give us **known ground truth** for the scanner (not for ML training):

- `vulnerable/`: one unsafe pattern per file, with the expected Bandit rule and line in a header comment.
- `safe/`: the fixed counterpart of each pattern.
- `edge/`: empty, binary, non-UTF-8, syntax errors, oversized, wrong file type.
- `adversarial/`: misleading comments and a canary file that proves submitted code is never executed.

In [5]:
fixtures = Path("../tests/fixtures")
print(pd.Series({d.name: len(list(d.glob("*"))) for d in sorted(fixtures.iterdir()) if d.is_dir()},
                name="files"))

adversarial     2
edge            7
safe           11
vulnerable     12
Name: files, dtype: int64


## Step 6: Comparison and the language decision

| | Devign | CVEfixes (Python) | Fixtures |
|---|---|---|---|
| Language | C | **Python** | **Python** |
| Unit | Function | Method (before/after fix) | File |
| Labels | Fix commits, manually checked | CVE fix commits | Written by us |
| Label noise | Medium–high | Medium | None |
| Groups | Commit | Repository | — |
| Used for | Practising the method (Exp. 02–03) | **Scanner ML model** (Exp. 04–05) | Scanner tests and final evaluation |

**Decision:** the scanner's ML model is trained on **Python** (CVEfixes) so that it reads the same language as Bandit. Devign stays in the project as an honestly reported C experiment. The application states which language each engine supports.